In [1]:
%cd ~/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
%load_ext autoreload
%autoreload 2
import sys
print(sys.executable)

/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室
/Users/leochen/Desktop/線上課程教材/pythan基礎觀念/爬蟲/好球帶研究室/.venv/bin/python


In [7]:
%%writefile scripts/lock_prediction.py
"""鎖定台灣大賽預測：在看到任何台灣大賽結果之前執行一次，存成 data/clean/prediction_locked.json。"""
import datetime as dt
import json
import sys
from pathlib import Path

ROOT = Path(__file__).resolve().parent.parent
sys.path.insert(0, str(ROOT))
import pandas as pd

from src import config, model

OUT = ROOT / "data/clean/prediction_locked.json"
if OUT.exists():
    sys.exit("已經鎖定過了，不能覆蓋。真的要重做，請先手動刪掉 data/clean/prediction_locked.json")

teams=pd.read_csv('data/clean/teams.csv').set_index('team')
A, B =config.TS_TEAMS
pa, pb=teams.loc[A,'pythag'] , teams.loc[B,'pythag']
p_a , results , _=model.simulate_series(pa,pb,config.TS_HOME_PATTERN,config.TS_NEED,config.HOME_ADVANTAGE)

order = ["4-0", "4-1", "4-2", "4-3", "3-4", "2-4", "1-4", "0-4"]
locked={
    "locked_at": dt .datetime.now().isoformat(timespec='seconds'),
    "teams":[A,B],
    "pythag":{A: round(pa,4), B: round(pb,4)},
    "home_pattern":config.TS_HOME_PATTERN,
    "home_advantage":config.HOME_ADVANTAGE,
    "game_prob_A_home":round(model.game_prob(pa ,pb,True,config.HOME_ADVANTAGE),4),
    "game_prob_A_away":round(model.game_prob(pa ,pb,False,config.HOME_ADVANTAGE),4),
    "champion_prob":{A: p_a, B: round(1- p_a ,4) },
    "result_dist":{ k: results.get(k,0)/10000 for k in order},
    "most_likely": max(results, key=results.get),
}
OUT.write_text(json.dumps(locked, ensure_ascii=False, indent=2), encoding="utf-8")
print(f"已鎖定：{A} 奪冠機率 {p_a:.1%}，最可能的比數 {locked['most_likely']}（{locked['locked_at']}）")

Writing scripts/lock_prediction.py


In [8]:
!python scripts/lock_prediction.py
!git add data/clean/prediction_locked.json
!git commit -m "鎖定台灣大賽預測"
!git tag prediction-locked

已鎖定：味全龍 奪冠機率 67.6%，最可能的比數 4-2（2026-10-29T18:51:57）
[main a6ee519] 鎖定台灣大賽預測
 Committer: Leo Chen <leochen@LeodeMacBook-Air.local>
Your name and email address were configured automatically based
on your username and hostname. Please check that they are accurate.
You can suppress this message by setting them explicitly. Run the
following command and follow the instructions in your editor to edit
your configuration file:

    git config --global --edit

After doing this, you may fix the identity used for this commit with:

    git commit --amend --reset-author

 1 file changed, 38 insertions(+)
 create mode 100644 data/clean/prediction_locked.json


In [12]:
%%writefile -a src/data.py
import json

def load_locked():
    """Day 18 鎖定的台灣大賽預測；還沒鎖定就回傳 None。"""
    path=CLEAN/'prediction_locked.json'
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else None


Appending to src/data.py
